In [ ]:
!pip install -q dspy-ai exa-py parallel-web gspread

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

print(" Successfully authenticated with Google Sheets!")

In [ ]:
import os
from getpass import getpass

class Config:
    """Central configuration. Secrets are loaded from environment variables."""

    # API Keys
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    EXA_API_KEY = os.getenv("EXA_API_KEY")
    PARALLEL_API_KEY = os.getenv("PARALLEL_API_KEY")

    # Model Configuration
    MODEL = "openrouter/anthropic/claude-sonnet-4.5"
    API_BASE = "https://openrouter.ai/api/v1"

    # Default Parameters
    WINDOW_DAYS = 90
    GEOGRAPHY = "EU US UK"
    BASELINE_DATE = "2024-01-01"

    # Limits
    MAX_TRIGGERS_PER_ENGINE = 20
    HOOKS_PER_TRIGGER = 3
    MAX_CEO_OPPORTUNITIES = 7

    # Parallel Processing
    MAX_WORKERS = 3

    # Google Sheets
    SHEET_NAME = "Target 25"
    INPUT_TAB = "Email Custom Fields - Ivan Prioritised Shortlist 1"
    OUTPUT_TABS = {
        "engine_a_g": "Engine_A_G_Triggers",
        "engine_a2_g": "Engine_A2_G_Triggers",
        "engine_b": "Engine_B_Hooks",
        "engine_c": "Engine_C_Refined_Hooks",
        "ceo_opportunities": "CEO_Opportunities"
    }


def require_secret(name: str) -> str:
    value = os.getenv(name)

    if not value:
        value = getpass(f"Enter {name}: ")
        os.environ[name] = value

    if not value.strip():
        raise ValueError(f"Missing required secret: {name}")

    return value


Config.OPENROUTER_API_KEY = require_secret("OPENROUTER_API_KEY")
Config.EXA_API_KEY = require_secret("EXA_API_KEY")
Config.PARALLEL_API_KEY = require_secret("PARALLEL_API_KEY")

print("✅ Configuration loaded!")
print(f"   • Window: {Config.WINDOW_DAYS} days")
print(f"   • Geography: {Config.GEOGRAPHY}")
print(f"   • Max workers: {Config.MAX_WORKERS}")
print(f"   • Sheet: {Config.SHEET_NAME}")
print("   • API keys: loaded securely, not printed")

In [ ]:
def initialize_apis():
    """Initialize all API clients"""
    print("🔧 Initializing APIs...")

    # DSPy LLM
    lm = dspy.LM(
        model=Config.MODEL,
        api_base=Config.API_BASE,
        api_key=Config.OPENROUTER_API_KEY,
    )
    dspy.configure(lm=lm)

    # Exa
    exa = Exa(api_key=Config.EXA_API_KEY)

    # Parallel
    parallel = Parallel(api_key=Config.PARALLEL_API_KEY)

    print("✅ APIs initialized successfully!")
    return exa, parallel, gc

exa, parallel, gc = initialize_apis()

In [ ]:
# Engine A-G: Direct asset triggers
class TriggerIdentification(dspy.Signature):
    """Identify material environmental and news triggers for an asset within specified timeframe"""
    asset: str = dspy.InputField(desc="Drug discovery asset name")
    indication: str = dspy.InputField(desc="Target indication")
    window_days: int = dspy.InputField(desc="Days to look back")
    geography: str = dspy.InputField(desc="Geographic scope (e.g., 'EU US UK')")
    baseline_date: str = dspy.InputField(desc="Events must be after this date")
    search_results: str = dspy.InputField(desc="Combined search results from all layers")

    triggers: str = dspy.OutputField(
        desc="List of up to 20 verified triggers in structured format. Each trigger must include: headline, date (ISO format), source, category, asset_relevance, delta_vs_baseline, impact_level (High/Medium/Low), citations. Return as JSON array."
    )

# Engine A2-G: Competitor/mechanism triggers
class CompetitorTriggerIdentification(dspy.Signature):
    """Identify competitor and mechanism-level environmental triggers"""
    asset: str = dspy.InputField(desc="Drug discovery asset name")
    indication: str = dspy.InputField(desc="Target indication")
    window_days: int = dspy.InputField(desc="Days to look back")
    geography: str = dspy.InputField(desc="Geographic scope")
    baseline_date: str = dspy.InputField(desc="Events must be after this date")
    search_results: str = dspy.InputField(desc="Combined competitor and mechanism search results")

    triggers: str = dspy.OutputField(
        desc="List of up to 20 verified competitor/mechanism triggers in structured format. Each trigger must include: headline, date (ISO format), source, category, competitor_or_class_relevance, delta_vs_baseline, impact_level (High/Medium/Low), citations. Return as JSON array."
    )

# Engine B: Hook generation
class HookGeneration(dspy.Signature):
    """Generate three CEO-level hooks for a single trigger"""
    asset: str = dspy.InputField(desc="Drug discovery asset name")
    indication: str = dspy.InputField(desc="Target indication")
    role: str = dspy.InputField(desc="Executive job title")
    trigger: str = dspy.InputField(desc="Single trigger in JSON format")

    hooks: str = dspy.OutputField(
        desc="Generate exactly 3 CEO-level hooks for this trigger. Each hook must: reference the asset by name, reference the core event, name an investor cohort (e.g., 'technical healthcare funds', 'fundamentals-oriented funds', 'growth-oriented investors'), use understated Evercore/Centerview advisory tone, be maximum 2 sentences, include light CTA ('if helpful', 'if of interest', 'if you want to see the detail'). Return as JSON array with keys: hook_1, hook_2, hook_3."
    )

# Engine C: Hook refinement
class HookRefinement(dspy.Signature):
    """Refine and calibrate CEO-level hooks to highest standard"""
    asset: str = dspy.InputField(desc="Drug discovery asset name")
    indication: str = dspy.InputField(desc="Target indication")
    role: str = dspy.InputField(desc="Executive job title")
    hooks_raw: str = dspy.InputField(desc="Raw hooks from Engine B in JSON format")

    hooks_refined: str = dspy.OutputField(
        desc="Refine each hook to senior-adviser standard. Maintain meaning while tightening tone, structure, investor framing and CTA. Use UK spelling. Maximum 2 sentences. Remove filler, reduce redundancy. Keep only high-signal phrasing. Senior adviser tone (Evercore/Centerview style): controlled, neutral, precise, quietly confident. Return as JSON array with keys: refined_hook_1, refined_hook_2, refined_hook_3."
    )

# Opportunity Engine
class CEOOpportunityIdentification(dspy.Signature):
    """Identify strategic opportunities and leverage shifts for CEO based on all triggers and hooks"""
    asset: str = dspy.InputField(desc="Drug discovery asset name")
    indication: str = dspy.InputField(desc="Target indication")
    all_triggers: str = dspy.InputField(desc="All triggers from Engine A-G and A2-G combined")
    all_hooks_refined: str = dspy.InputField(desc="All refined hooks from Engine C")

    opportunities: str = dspy.OutputField(
        desc="Identify 5-7 CEO-level opportunities. For each: title (one short line), description (2-4 lines explaining what changed in environment), why_it_matters (2-3 lines explaining what this enables). Focus on: credibility and narrative control, negotiating position and leverage, optionality and strategic windows, investor base depth and quality, long-term value arc and durability. Use UK spelling. Return as JSON array with keys: title, description, why_it_matters."
    )

print("✅ DSPy signatures defined!")

In [ ]:
# ============================================
# ENGINE A-G: Direct Asset Triggers
# ============================================

class EngineAG:
    """Engine A-G: Environmental and news change detection for specified asset"""

    def __init__(self, exa: Exa, parallel: Parallel):
        self.exa = exa
        self.parallel = parallel
        self.trigger_identifier = dspy.ChainOfThought(TriggerIdentification)

    def search_general_news(self, asset: str, company: str, indication: str, start_date: str) -> str:
        """Search broad news and web sources"""
        print("    📰 Searching general news layer...")
        results = []

        queries = [
            f"{asset} {company} {indication}",
            f"{asset} clinical trial",
            f"{asset} readout",
            f"{indication} treatment update"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=5,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:2000]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def search_company_ir(self, company: str) -> str:
        """Search company IR and partner sites"""
        print("    🏢 Searching company/partner IR layer...")
        results = []

        try:
            # Find company website
            website_search = self.parallel.beta.search(
                objective=f"Find the official website URL for {company} biotech/pharmaceutical company"
            )

            # Find PR/News page
            pr_search = self.parallel.beta.search(
                objective=f"Find the press releases or news announcements page for {company} biotech company"
            )

            # Extract PR content
            if pr_search.results:
                pr_urls = [r.url for r in pr_search.results[:2] if r.url]
                if pr_urls:
                    try:
                        extract_result = self.parallel.beta.extract(
                            urls=pr_urls,
                            excerpts=True,
                            full_content=True
                        )
                        results.append(f"PR Content: {str(extract_result.results)[:2000]}")
                    except:
                        results.append(f"PR URLs: {pr_urls}")

            return "\n".join(results) if results else "No company IR results found"
        except Exception as e:
            return f"Company IR search error: {str(e)}"

    def search_regulatory_trial(self, asset: str, indication: str, start_date: str) -> str:
        """Search regulatory and trial registries"""
        print("    ⚖️ Searching regulatory/trial layer...")
        results = []

        queries = [
            f"{asset} FDA approval",
            f"{asset} EMA approval",
            f"{asset} clinical trial update",
            f"{asset} {indication} trial status"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=3,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:1500]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def search_scientific(self, asset: str, indication: str, start_date: str) -> str:
        """Search journals and conference abstracts"""
        print("    🔬 Searching scientific layer...")
        results = []

        queries = [
            f"{asset} clinical data publication",
            f"{asset} conference abstract",
            f"{indication} {asset} journal"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=3,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:1500]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def run(self, asset: str, company: str, indication: str,
            window_days: int, geography: str, baseline_date: str) -> List[Dict]:
        """Run complete Engine A-G pipeline"""
        print(f"\n🎯 ENGINE A-G: {asset} / {indication}")

        # Calculate date range
        start_date = (datetime.now() - timedelta(days=window_days)).strftime("%Y-%m-%d")

        # Run all search layers
        general_news = self.search_general_news(asset, company, indication, start_date)
        company_ir = self.search_company_ir(company)
        regulatory = self.search_regulatory_trial(asset, indication, start_date)
        scientific = self.search_scientific(asset, indication, start_date)

        # Combine all search results
        combined_results = f"""
=== GENERAL NEWS LAYER ===
{general_news}

=== COMPANY IR LAYER ===
{company_ir}

=== REGULATORY/TRIAL LAYER ===
{regulatory}

=== SCIENTIFIC LAYER ===
{scientific}
"""

        # Identify triggers using LLM
        print("    🤖 Identifying triggers with LLM...")
        try:
            result = self.trigger_identifier(
                asset=asset,
                indication=indication,
                window_days=window_days,
                geography=geography,
                baseline_date=baseline_date,
                search_results=combined_results
            )

            # Parse triggers
            # Parse triggers with better error handling
            triggers_json = result.triggers
            print(f"    🔍 LLM Response (first 500 chars): {str(triggers_json)[:500]}")

            try:
                if isinstance(triggers_json, str):
                    # Try to extract JSON if wrapped in markdown
                    if "```json" in triggers_json:
                        triggers_json = triggers_json.split("```json")[1].split("```")[0].strip()
                    elif "```" in triggers_json:
                        triggers_json = triggers_json.split("```")[1].split("```")[0].strip()

                    triggers = json.loads(triggers_json)
                else:
                    triggers = triggers_json

                # Ensure it's a list
                if isinstance(triggers, dict):
                    triggers = [triggers]
                elif not isinstance(triggers, list):
                    print(f"    ⚠️ Unexpected triggers format: {type(triggers)}")
                    triggers = []

            except json.JSONDecodeError as e:
                print(f"    ⚠️ JSON parse error: {e}")
                print(f"    Raw response: {triggers_json[:1000]}")
                triggers = []

            # Limit to max triggers
            if len(triggers) > Config.MAX_TRIGGERS_PER_ENGINE:
                triggers = triggers[:Config.MAX_TRIGGERS_PER_ENGINE]

            print(f"    ✅ Found {len(triggers)} triggers")
            return triggers

        except Exception as e:
            print(f"    ❌ Error identifying triggers: {str(e)}")
            return []


# ============================================
# ENGINE A2-G: Competitor/Mechanism Triggers
# ============================================

class EngineA2G:
    """Engine A2-G: Competitor and mechanism-level intelligence"""

    def __init__(self, exa: Exa, parallel: Parallel):
        self.exa = exa
        self.parallel = parallel
        self.trigger_identifier = dspy.ChainOfThought(CompetitorTriggerIdentification)

    def search_competitor_news(self, indication: str, start_date: str) -> str:
        """Search competitor and mechanism-level events"""
        print("    🏁 Searching competitor news layer...")
        results = []

        queries = [
            f"{indication} competitor trial",
            f"{indication} mechanism breakthrough",
            f"{indication} inhibitor update",
            f"{indication} competitor readout"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=5,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:2000]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def search_mechanism_scientific(self, indication: str, start_date: str) -> str:
        """Search mechanism/class-level insights"""
        print("    🧬 Searching mechanism/scientific layer...")
        results = []

        queries = [
            f"{indication} mechanism of action",
            f"{indication} resistance findings",
            f"{indication} biomarker discovery"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=3,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:1500]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def search_competitor_regulatory(self, indication: str, start_date: str) -> str:
        """Search regulatory shifts in competitive landscape"""
        print("    📋 Searching competitor regulatory layer...")
        results = []

        queries = [
            f"{indication} FDA approval competitor",
            f"{indication} regulatory guideline update"
        ]

        for query in queries:
            try:
                exa_result = self.exa.search_and_contents(
                    query,
                    text=True,
                    type="auto",
                    num_results=3,
                    start_published_date=start_date
                )
                results.append(f"Query: {query}\nResults: {str(exa_result)[:1500]}\n")
            except Exception as e:
                results.append(f"Query: {query}\nError: {str(e)}\n")

        return "\n".join(results)

    def search_competitor_commercial(self, indication: str) -> str:
        """Search commercial shifts"""
        print("    💼 Searching competitor commercial layer...")
        results = []

        try:
            parallel_result = self.parallel.beta.search(
                objective=f"Find recent {indication} competitor licensing deals, M&A, and financing"
            )

            if parallel_result.results:
                for r in parallel_result.results[:5]:
                    results.append(f"Title: {r.title}\nURL: {r.url}\nExcerpts: {str(r.excerpts)[:500]}\n")

            return "\n".join(results) if results else "No commercial results found"
        except Exception as e:
            return f"Commercial search error: {str(e)}"

    def run(self, asset: str, indication: str, window_days: int,
            geography: str, baseline_date: str) -> List[Dict]:
        """Run complete Engine A2-G pipeline"""
        print(f"\n🔍 ENGINE A2-G: Competitor/Mechanism Analysis for {indication}")

        # Calculate date range
        start_date = (datetime.now() - timedelta(days=window_days)).strftime("%Y-%m-%d")

        # Run all search layers
        competitor_news = self.search_competitor_news(indication, start_date)
        mechanism_sci = self.search_mechanism_scientific(indication, start_date)
        competitor_reg = self.search_competitor_regulatory(indication, start_date)
        competitor_comm = self.search_competitor_commercial(indication)

        # Combine all search results
        combined_results = f"""
=== COMPETITOR NEWS LAYER ===
{competitor_news}

=== MECHANISM/SCIENTIFIC LAYER ===
{mechanism_sci}

=== COMPETITOR REGULATORY LAYER ===
{competitor_reg}

=== COMPETITOR COMMERCIAL LAYER ===
{competitor_comm}
"""

        # Identify triggers using LLM
        print("    🤖 Identifying competitor/mechanism triggers with LLM...")
        try:
            result = self.trigger_identifier(
                asset=asset,
                indication=indication,
                window_days=window_days,
                geography=geography,
                baseline_date=baseline_date,
                search_results=combined_results
            )

            # Parse triggers with better error handling
            triggers_json = result.triggers
            print(f"    🔍 LLM Response (first 500 chars): {str(triggers_json)[:500]}")

            try:
                if isinstance(triggers_json, str):
                    # Try to extract JSON if wrapped in markdown
                    if "```json" in triggers_json:
                        triggers_json = triggers_json.split("```json")[1].split("```")[0].strip()
                    elif "```" in triggers_json:
                        triggers_json = triggers_json.split("```")[1].split("```")[0].strip()

                    triggers = json.loads(triggers_json)
                else:
                    triggers = triggers_json

                # Ensure it's a list
                if isinstance(triggers, dict):
                    triggers = [triggers]
                elif not isinstance(triggers, list):
                    print(f"    ⚠️ Unexpected triggers format: {type(triggers)}")
                    triggers = []

            except json.JSONDecodeError as e:
                print(f"    ⚠️ JSON parse error: {e}")
                print(f"    Raw response: {triggers_json[:1000]}")
                triggers = []

            # Limit to max triggers
            if len(triggers) > Config.MAX_TRIGGERS_PER_ENGINE:
                triggers = triggers[:Config.MAX_TRIGGERS_PER_ENGINE]

            print(f"    ✅ Found {len(triggers)} competitor/mechanism triggers")
            return triggers

        except Exception as e:
            print(f"    ❌ Error identifying triggers: {str(e)}")
            return []


# ============================================
# ENGINE B: Hook Generation
# ============================================

class EngineB:
    """Engine B: CEO-level hook generation"""

    def __init__(self):
        self.hook_generator = dspy.ChainOfThought(HookGeneration)

    def run(self, asset: str, indication: str, role: str,
            triggers_a_g: List[Dict], triggers_a2_g: List[Dict]) -> List[Dict]:
        """Generate 3 hooks per trigger"""
        print(f"\n✍️ ENGINE B: Generating CEO-level hooks")

        # Combine all triggers
        all_triggers = triggers_a_g + triggers_a2_g

        if not all_triggers:
            print("    ⚠️ No triggers to process")
            return []

        all_hooks = []

        for idx, trigger in enumerate(all_triggers):
            print(f"    Hook generation for trigger {idx + 1}/{len(all_triggers)}...")
            try:
                result = self.hook_generator(
                    asset=asset,
                    indication=indication,
                    role=role,
                    trigger=json.dumps(trigger)
                )

                # Parse hooks
                hooks_json = result.hooks
                hooks = json.loads(hooks_json) if isinstance(hooks_json, str) else hooks_json

                all_hooks.append({
                    "trigger": trigger,
                    "hooks": hooks
                })

            except Exception as e:
                print(f"    ❌ Error generating hooks for trigger {idx + 1}: {str(e)}")
                all_hooks.append({
                    "trigger": trigger,
                    "hooks": {"error": str(e)}
                })

        print(f"    ✅ Generated hooks for {len(all_hooks)} triggers")
        return all_hooks


# ============================================
# ENGINE C: Hook Refinement
# ============================================

class EngineC:
    """Engine C: Hook refinement and calibration"""

    def __init__(self):
        self.hook_refiner = dspy.ChainOfThought(HookRefinement)

    def run(self, asset: str, indication: str, role: str,
            hooks_raw: List[Dict]) -> List[Dict]:
        """Refine all hooks to highest standard"""
        print(f"\n🎨 ENGINE C: Refining hooks to senior-adviser standard")

        if not hooks_raw:
            print("    ⚠️ No hooks to refine")
            return []

        refined_hooks_all = []

        for idx, hook_set in enumerate(hooks_raw):
            print(f"    Refining hook set {idx + 1}/{len(hooks_raw)}...")
            try:
                result = self.hook_refiner(
                    asset=asset,
                    indication=indication,
                    role=role,
                    hooks_raw=json.dumps(hook_set["hooks"])
                )

                # Parse refined hooks
                refined_json = result.hooks_refined
                refined = json.loads(refined_json) if isinstance(refined_json, str) else refined_json

                refined_hooks_all.append({
                    "trigger": hook_set["trigger"],
                    "hooks_raw": hook_set["hooks"],
                    "hooks_refined": refined
                })

            except Exception as e:
                print(f"    ❌ Error refining hooks for set {idx + 1}: {str(e)}")
                refined_hooks_all.append({
                    "trigger": hook_set["trigger"],
                    "hooks_raw": hook_set["hooks"],
                    "hooks_refined": {"error": str(e)}
                })

        print(f"    ✅ Refined {len(refined_hooks_all)} hook sets")
        return refined_hooks_all


# ============================================
# CEO OPPORTUNITY ENGINE
# ============================================

class CEOOpportunityEngine:
    """CEO Opportunity Engine: Strategic leverage identification"""

    def __init__(self):
        self.opportunity_identifier = dspy.ChainOfThought(CEOOpportunityIdentification)

    def run(self, asset: str, indication: str,
            triggers_a_g: List[Dict], triggers_a2_g: List[Dict],
            hooks_refined: List[Dict]) -> List[Dict]:
        """Identify CEO-level strategic opportunities"""
        print(f"\n🎯 CEO OPPORTUNITY ENGINE: Identifying strategic opportunities")

        # Combine all triggers
        all_triggers = triggers_a_g + triggers_a2_g

        if not all_triggers:
            print("    ⚠️ No triggers for opportunity analysis")
            return []

        try:
            result = self.opportunity_identifier(
                asset=asset,
                indication=indication,
                all_triggers=json.dumps(all_triggers),
                all_hooks_refined=json.dumps(hooks_refined)
            )

            # Parse opportunities
            opportunities_json = result.opportunities
            opportunities = json.loads(opportunities_json) if isinstance(opportunities_json, str) else opportunities_json

            # Limit to max opportunities
            if len(opportunities) > Config.MAX_CEO_OPPORTUNITIES:
                opportunities = opportunities[:Config.MAX_CEO_OPPORTUNITIES]

            print(f"    ✅ Identified {len(opportunities)} CEO opportunities")
            return opportunities

        except Exception as e:
            print(f"    ❌ Error identifying opportunities: {str(e)}")
            return []


# ============================================
# GOOGLE SHEETS MANAGER
# ============================================

class GoogleSheetsManager:
    """Manage Google Sheets input/output"""

    def __init__(self, gc: gspread.Client):
        self.gc = gc
        self.sheet = gc.open(Config.SHEET_NAME)
        self.ensure_output_tabs()

    def ensure_output_tabs(self):
        """Create output tabs if they don't exist"""
        print("📊 Ensuring output tabs exist...")
        existing_tabs = [ws.title for ws in self.sheet.worksheets()]

        for tab_key, tab_name in Config.OUTPUT_TABS.items():
            if tab_name not in existing_tabs:
                print(f"    Creating tab: {tab_name}")
                self.sheet.add_worksheet(title=tab_name, rows=1000, cols=50)

    def read_input_data(self) -> List[Dict]:
      """Read prospect data from input tab"""
      print(f" Reading input data from '{Config.INPUT_TAB}'...")
      worksheet = self.sheet.worksheet(Config.INPUT_TAB)
      all_data = worksheet.get_all_records()

      # Filter out already processed rows (those with Processing_Status set)
      data = []
      for row in all_data:
          status = row.get('Processing_Status', '')
          # Only include rows that haven't been processed yet
          if not status or status == '':
              data.append(row)

      print(f"    ✅ Found {len(data)} unprocessed prospects (out of {len(all_data)} total)")
      return data

    def write_engine_a_g_triggers(self, row_num: int, prospect: Dict, triggers: List[Dict]):
        """Write Engine A-G triggers to dedicated tab"""
        worksheet = self.sheet.worksheet(Config.OUTPUT_TABS["engine_a_g"])

        # Write headers if first row
        if row_num == 0:
            headers = ["Row", "Company", "Asset", "Indication", "Trigger_Index",
                       "Headline", "Date", "Source", "Category", "Asset_Relevance",
                       "Delta_vs_Baseline", "Impact_Level", "Citations"]
            worksheet.update('A1:M1', [headers])

        # Write triggers
        for idx, trigger in enumerate(triggers):
            row_data = [
                row_num + 2,
                prospect.get("Company Name", ""),
                prospect.get("Asset Name", ""),
                prospect.get("Indication", ""),
                idx + 1,
                trigger.get("headline", ""),
                trigger.get("date", ""),
                trigger.get("source", ""),
                trigger.get("category", ""),
                trigger.get("asset_relevance", ""),
                trigger.get("delta_vs_baseline", ""),
                trigger.get("impact_level", ""),
                "; ".join(trigger.get("citations", [])) if isinstance(trigger.get("citations"), list) else trigger.get("citations", "")
            ]
            worksheet.append_row(row_data)

    def write_engine_a2_g_triggers(self, row_num: int, prospect: Dict, triggers: List[Dict]):
        """Write Engine A2-G triggers to dedicated tab"""
        worksheet = self.sheet.worksheet(Config.OUTPUT_TABS["engine_a2_g"])

        # Write headers if first row
        if row_num == 0:
            headers = ["Row", "Company", "Asset", "Indication", "Trigger_Index",
                       "Headline", "Date", "Source", "Category", "Competitor_Class_Relevance",
                       "Delta_vs_Baseline", "Impact_Level", "Citations"]
            worksheet.update('A1:M1', [headers])

        # Write triggers
        for idx, trigger in enumerate(triggers):
            row_data = [
                row_num + 2,
                prospect.get("Company Name", ""),
                prospect.get("Asset Name", ""),
                prospect.get("Indication", ""),
                idx + 1,
                trigger.get("headline", ""),
                trigger.get("date", ""),
                trigger.get("source", ""),
                trigger.get("category", ""),
                trigger.get("competitor_or_class_relevance", ""),
                trigger.get("delta_vs_baseline", ""),
                trigger.get("impact_level", ""),
                "; ".join(trigger.get("citations", [])) if isinstance(trigger.get("citations"), list) else trigger.get("citations", "")
            ]
            worksheet.append_row(row_data)

    def write_engine_b_hooks(self, row_num: int, prospect: Dict, hooks: List[Dict]):
        """Write Engine B hooks to dedicated tab"""
        worksheet = self.sheet.worksheet(Config.OUTPUT_TABS["engine_b"])

        # Write headers if first row
        if row_num == 0:
            headers = ["Row", "Company", "Asset", "Indication", "Trigger_Index",
                       "Trigger_Headline", "Hook_1", "Hook_2", "Hook_3"]
            worksheet.update('A1:I1', [headers])

        # Write hooks
        for idx, hook_set in enumerate(hooks):
            hooks_data = hook_set.get("hooks", {})
            row_data = [
                row_num + 2,
                prospect.get("Company Name", ""),
                prospect.get("Asset Name", ""),
                prospect.get("Indication", ""),
                idx + 1,
                hook_set.get("trigger", {}).get("headline", ""),
                hooks_data.get("hook_1", hooks_data.get("error", "")),
                hooks_data.get("hook_2", ""),
                hooks_data.get("hook_3", "")
            ]
            worksheet.append_row(row_data)

    def write_engine_c_refined_hooks(self, row_num: int, prospect: Dict, refined_hooks: List[Dict]):
        """Write Engine C refined hooks to dedicated tab"""
        worksheet = self.sheet.worksheet(Config.OUTPUT_TABS["engine_c"])

        # Write headers if first row
        if row_num == 0:
            headers = ["Row", "Company", "Asset", "Indication", "Trigger_Index",
                       "Trigger_Headline", "Refined_Hook_1", "Refined_Hook_2", "Refined_Hook_3"]
            worksheet.update('A1:I1', [headers])

        # Write refined hooks
        for idx, hook_set in enumerate(refined_hooks):
            refined_data = hook_set.get("hooks_refined", {})
            row_data = [
                row_num + 2,
                prospect.get("Company Name", ""),
                prospect.get("Asset Name", ""),
                prospect.get("Indication", ""),
                idx + 1,
                hook_set.get("trigger", {}).get("headline", ""),
                refined_data.get("refined_hook_1", refined_data.get("error", "")),
                refined_data.get("refined_hook_2", ""),
                refined_data.get("refined_hook_3", "")
            ]
            worksheet.append_row(row_data)

    def write_ceo_opportunities(self, row_num: int, prospect: Dict, opportunities: List[Dict]):
        """Write CEO opportunities to dedicated tab"""
        worksheet = self.sheet.worksheet(Config.OUTPUT_TABS["ceo_opportunities"])

        # Write headers if first row
        if row_num == 0:
            headers = ["Row", "Company", "Asset", "Indication", "Opportunity_Index",
                       "Title", "Description", "Why_It_Matters"]
            worksheet.update('A1:H1', [headers])

        # Write opportunities
        for idx, opp in enumerate(opportunities):
            row_data = [
                row_num + 2,
                prospect.get("Company Name", ""),
                prospect.get("Asset Name", ""),
                prospect.get("Indication", ""),
                idx + 1,
                opp.get("title", ""),
                opp.get("description", ""),
                opp.get("why_it_matters", "")
            ]
            worksheet.append_row(row_data)

    def write_summary_to_main(self, row_num: int, trigger_count_a_g: int,
                               trigger_count_a2_g: int, hook_count: int,
                               opp_count: int, status: str, error: str = ""):
        """Write summary statistics to main input tab"""
        worksheet = self.sheet.worksheet(Config.INPUT_TAB)

        # Find or create summary columns
        headers = worksheet.row_values(1)

        summary_cols = {
            "Engine_A_G_Triggers": None,
            "Engine_A2_G_Triggers": None,
            "Total_Hooks": None,
            "CEO_Opportunities": None,
            "Processing_Status": None,
            "Error_Log": None
        }

        # Find existing columns or add new ones
        for col_name in summary_cols.keys():
            if col_name in headers:
                summary_cols[col_name] = headers.index(col_name) + 1
            else:
                # Add new column
                new_col_idx = len(headers) + 1
                worksheet.update_cell(1, new_col_idx, col_name)
                summary_cols[col_name] = new_col_idx
                headers.append(col_name)

        # Write summary data
        worksheet.update_cell(row_num + 2, summary_cols["Engine_A_G_Triggers"], trigger_count_a_g)
        worksheet.update_cell(row_num + 2, summary_cols["Engine_A2_G_Triggers"], trigger_count_a2_g)
        worksheet.update_cell(row_num + 2, summary_cols["Total_Hooks"], hook_count)
        worksheet.update_cell(row_num + 2, summary_cols["CEO_Opportunities"], opp_count)
        worksheet.update_cell(row_num + 2, summary_cols["Processing_Status"], status)
        if error:
            worksheet.update_cell(row_num + 2, summary_cols["Error_Log"], error)


# ============================================
# PROSPECT PROCESSOR
# ============================================

def process_single_prospect(prospect: Dict, row_num: int,
                            engine_a_g: EngineAG, engine_a2_g: EngineA2G,
                            engine_b: EngineB, engine_c: EngineC,
                            ceo_engine: CEOOpportunityEngine,
                            sheets_manager: GoogleSheetsManager) -> Dict:
    """Process a single prospect through all engines"""

    print(f"\n{'='*80}")
    print(f"🚀 PROCESSING PROSPECT {row_num + 1}")
    print(f"{'='*80}")

    # DEBUG: Print the entire prospect dictionary
    print(f"\n🔍 DEBUG - Prospect type: {type(prospect)}")
    print(f"🔍 DEBUG - Prospect keys: {list(prospect.keys())}")
    print(f"🔍 DEBUG - Full prospect data:")
    for key, value in prospect.items():
        print(f"     '{key}': '{value}'")

    result = {
        "row_num": row_num,
        "prospect": prospect,
        "status": "processing",
        "error": ""
    }

    try:
        # Extract prospect data with DEBUG
        print(f"\n🔍 Extracting fields...")
        asset = prospect.get("Asset Name", "")
        print(f"   Asset Name: '{asset}' (found: {bool(asset)})")

        company = prospect.get("Company Name", "")
        print(f"   Company Name: '{company}' (found: {bool(company)})")

        indication = prospect.get("Indication", "")
        print(f"   Indication: '{indication}' (found: {bool(indication)})")

        role = prospect.get("Job Title", "")
        print(f"   Job Title: '{role}' (found: {bool(role)})")

        # Check if required fields exist
        print(f"\n🔍 Validation check:")
        print(f"   all([asset, company, indication]) = {all([asset, company, indication])}")
        print(f"   asset: {repr(asset)}")
        print(f"   company: {repr(company)}")
        print(f"   indication: {repr(indication)}")

        if not all([asset, company, indication]):
            result["status"] = "skipped"
            result["error"] = "Missing required fields (Asset Name, Company Name, or Indication)"
            print(f"⚠️ Skipping: {result['error']}")
            return result

        print(f"✅ All required fields present!")
        print(f"📋 Asset: {asset}")
        print(f"🏢 Company: {company}")
        print(f"🎯 Indication: {indication}")

        # Run Engine A-G
        triggers_a_g = engine_a_g.run(
            asset=asset,
            company=company,
            indication=indication,
            window_days=Config.WINDOW_DAYS,
            geography=Config.GEOGRAPHY,
            baseline_date=Config.BASELINE_DATE
        )
        result["triggers_a_g"] = triggers_a_g

        # Run Engine A2-G
        triggers_a2_g = engine_a2_g.run(
            asset=asset,
            indication=indication,
            window_days=Config.WINDOW_DAYS,
            geography=Config.GEOGRAPHY,
            baseline_date=Config.BASELINE_DATE
        )
        result["triggers_a2_g"] = triggers_a2_g

        # Run Engine B
        hooks_raw = engine_b.run(
            asset=asset,
            indication=indication,
            role=role,
            triggers_a_g=triggers_a_g,
            triggers_a2_g=triggers_a2_g
        )
        result["hooks_raw"] = hooks_raw

        # Run Engine C
        hooks_refined = engine_c.run(
            asset=asset,
            indication=indication,
            role=role,
            hooks_raw=hooks_raw
        )
        result["hooks_refined"] = hooks_refined

        # Run CEO Opportunity Engine
        opportunities = ceo_engine.run(
            asset=asset,
            indication=indication,
            triggers_a_g=triggers_a_g,
            triggers_a2_g=triggers_a2_g,
            hooks_refined=hooks_refined
        )
        result["opportunities"] = opportunities

        # Write all outputs to Google Sheets
        print(f"\n💾 Writing results to Google Sheets...")
        sheets_manager.write_engine_a_g_triggers(row_num, prospect, triggers_a_g)
        sheets_manager.write_engine_a2_g_triggers(row_num, prospect, triggers_a2_g)
        sheets_manager.write_engine_b_hooks(row_num, prospect, hooks_raw)
        sheets_manager.write_engine_c_refined_hooks(row_num, prospect, hooks_refined)
        sheets_manager.write_ceo_opportunities(row_num, prospect, opportunities)

        # Write summary to main tab
        sheets_manager.write_summary_to_main(
            row_num=row_num,
            trigger_count_a_g=len(triggers_a_g),
            trigger_count_a2_g=len(triggers_a2_g),
            hook_count=len(hooks_raw),
            opp_count=len(opportunities),
            status="completed"
        )

        result["status"] = "completed"
        print(f"✅ COMPLETED PROSPECT {row_num + 1}")

    except Exception as e:
        error_msg = f"Error: {str(e)}\n{traceback.format_exc()}"
        result["status"] = "error"
        result["error"] = error_msg
        print(f"❌ ERROR processing prospect: {error_msg}")

        # Write error to main tab
        try:
            sheets_manager.write_summary_to_main(
                row_num=row_num,
                trigger_count_a_g=0,
                trigger_count_a2_g=0,
                hook_count=0,
                opp_count=0,
                status="error",
                error=str(e)
            )
        except:
            pass

    return result

print("All engine classes loaded!")


## Step 7: Main Execution

In [ ]:
def main():
    """Main execution function"""

    print("\n" + "="*80)
    print("🎯 HELIX CEO INTELLIGENCE ENGINE")
    print("="*80 + "\n")

    # Initialize engines
    print("🔧 Initializing engines...")
    engine_a_g = EngineAG(exa, parallel)
    engine_a2_g = EngineA2G(exa, parallel)
    engine_b = EngineB()
    engine_c = EngineC()
    ceo_engine = CEOOpportunityEngine()
    print("✅ All engines initialized!")

    # Initialize Google Sheets manager
    sheets_manager = GoogleSheetsManager(gc)

    # Read input data
    prospects = sheets_manager.read_input_data()

    if not prospects:
        print("⚠️ No prospects found in input sheet!")
        return

    print(f"\n📊 Configuration:")
    print(f"   • Window: {Config.WINDOW_DAYS} days")
    print(f"   • Geography: {Config.GEOGRAPHY}")
    print(f"   • Baseline: {Config.BASELINE_DATE}")
    print(f"   • Max workers: {Config.MAX_WORKERS}")
    print(f"   • Prospects available: {len(prospects)}")

    # Ask user how many to process
    print(f"\n❓ How many prospects would you like to process? (1-{len(prospects)})")
    print(f"   Enter number or 'all': ", end="")

    user_input = input().strip().lower()
    if user_input == 'all':
        num_to_process = len(prospects)
    else:
        try:
            num_to_process = min(int(user_input), len(prospects))
        except:
            num_to_process = 1

    prospects_to_process = prospects[:num_to_process]

    print(f"\n🚀 Processing {num_to_process} prospect(s) with {Config.MAX_WORKERS} parallel workers...")
    print(f"⏱️ Estimated time: {num_to_process * 10 // max(Config.MAX_WORKERS, 1)} - {num_to_process * 15 // max(Config.MAX_WORKERS, 1)} minutes\n")

    start_time = time.time()

    # Process prospects in parallel
    results = []
    with ThreadPoolExecutor(max_workers=Config.MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                process_single_prospect,
                prospect, idx,
                engine_a_g, engine_a2_g, engine_b, engine_c, ceo_engine,
                sheets_manager
            ): idx
            for idx, prospect in enumerate(prospects_to_process)
        }

        for future in as_completed(futures):
            result = future.result()
            results.append(result)

    # Summary
    elapsed_time = time.time() - start_time
    completed = sum(1 for r in results if r["status"] == "completed")
    errors = sum(1 for r in results if r["status"] == "error")
    skipped = sum(1 for r in results if r["status"] == "skipped")

    print(f"\n" + "="*80)
    print("📊 PROCESSING COMPLETE")
    print("="*80)
    print(f"✅ Completed: {completed}")
    print(f"❌ Errors: {errors}")
    print(f"⏭️ Skipped: {skipped}")
    print(f"⏱️ Total time: {elapsed_time/60:.1f} minutes")
    if num_to_process > 0:
        print(f"📈 Average: {elapsed_time/num_to_process:.1f} seconds per prospect")
    print(f"\n🎉 Results written to Google Sheet: '{Config.SHEET_NAME}'")
    print("="*80 + "\n")

# Run the main function
main()